# Q1 — Gujarati N-gram Language Models with Add-1 (Laplace) Smoothing


Q1. Take the tokenized Gujarati data from Assignment-1 and build four language models:
- Unigram
- Bigram
- Trigram
- Quadrigram

Use at least 1,000,000 sentences and split them into:
- Training: remaining sentences
- Development: 1,000 sentences
- Test: 1,000 sentences

This notebook calculates:
- Next-word prediction accuracy for Bigram, Trigram and Quadrigram models
- Perplexity for Unigram, Bigram, Trigram and Quadrigram models
- Add-1 (Laplace) smoothed probabilities

No pickle/model files are saved.


In [1]:
import math
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

In [2]:
DATA_PATH = Path(
    r"C:/Users/VIVEK/Desktop/NLP Lab/ASSIGNMENT_1/Q1/tokenized_gujarati_corpus/10lakh sentence.parquet"
)

df = pd.read_parquet(DATA_PATH, columns=["tokens"])

print("Total sentences:", len(df))

Total sentences: 1099721


In [3]:
print(df.head())
print(df["tokens"].iloc[0])
print("Tokens in first sentence:", len(df["tokens"].iloc[0]))

                                              tokens
0  [આ, વીડિયો, જુઓ, :, ઊંઝા, માર્કેટયાર્ડ, આજથી, ...
1                       [મિથેનોલ, આવ્યો, ક્યાંથી, ?]
2  [આખરે, ત્રણ, રાજ્યોમાં, મળેલ, હાર, પર, કોંગ્રે...
3  [તેમણે, કહ્યું, કે, ,, ત્રિપુરા, ,, નાગાલેન્ડ,...
4  [આ, આંકડો, માટે, ,, અને, વજન, ઘટાડવા, માટે, પ્...
['આ' 'વીડિયો' 'જુઓ' ':' 'ઊંઝા' 'માર્કેટયાર્ડ' 'આજથી' '25' 'જુલાઈ' 'સુધી'
 'બંધ']
Tokens in first sentence: 11


In [ ]:
# Shuffle the complete dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# 1,000 development + 1,000 test + all remaining sentences for training
dev_df = df.iloc[:1000].copy()
test_df = df.iloc[1000:2000].copy()
train_df = df.iloc[2000:].copy()

print("Total sentences:", len(df))
print("Training sentences:", len(train_df))
print("Development sentences:", len(dev_df))
print("Test sentences:", len(test_df))

if len(df) < 1_000_000:
    print("WARNING: Fewer than 1,000,000 sentences are available.")
else:
    print("Requirement satisfied: at least 1,000,000 sentences.")


Total sentences: 1099721
Training sentences: 1097721
Development sentences: 1000
Test sentences: 1000
Requirement satisfied: at least 1,000,000 sentences.


In [5]:
def most_common(counter):
    
    if not counter:
        return None
    
    return counter.most_common(1)[0][0]

In [ ]:
def train_model(sentences):
    unigram = Counter()
    bigram = defaultdict(Counter)
    trigram = defaultdict(Counter)
    quadrigram = defaultdict(Counter)

    for tokens in sentences:
        tokens = list(tokens)

        # Unigram counts
        for word in tokens:
            unigram[word] += 1

        # Count sentence end
        unigram["</s>"] += 1

        # Bigram counts
        tokens2 = ["<s>"] + tokens + ["</s>"]

        for i in range(len(tokens2) - 1):
            w1 = tokens2[i]
            w2 = tokens2[i + 1]
            bigram[w1][w2] += 1

        # Trigram counts
        tokens3 = ["<s>", "<s>"] + tokens + ["</s>"]

        for i in range(len(tokens3) - 2):
            w1 = tokens3[i]
            w2 = tokens3[i + 1]
            w3 = tokens3[i + 2]
            trigram[(w1, w2)][w3] += 1

        # Quadrigram counts
        tokens4 = ["<s>", "<s>", "<s>"] + tokens + ["</s>"]

        for i in range(len(tokens4) - 3):
            w1 = tokens4[i]
            w2 = tokens4[i + 1]
            w3 = tokens4[i + 2]
            w4 = tokens4[i + 3]
            quadrigram[(w1, w2, w3)][w4] += 1

    return unigram, bigram, trigram, quadrigram


In [7]:
unigram, bigram, trigram, quadrigram = train_model(
    train_df["tokens"]
)

# Vocabulary used for predicting the next word.
V = len(unigram) - 1

print("Prediction vocabulary size:", V)
print("Number of unigram entries:", len(unigram))
print("Number of bigram contexts:", len(bigram))
print("Number of trigram contexts:", len(trigram))
print("Number of quadrigram contexts:", len(quadrigram))


Prediction vocabulary size: 514766
Number of unigram entries: 514767
Number of bigram contexts: 514767
Number of trigram contexts: 5453509
Number of quadrigram contexts: 10690894


In [8]:
print("Top 20 most frequent words:\n")

for word, count in unigram.most_common(20):

    print(
        word,
        "->",
        count
    )

Top 20 most frequent words:

</s> -> 1097721
. -> 946503
છે -> 610887
, -> 448074
અને -> 277336
આ -> 179705
કે -> 158481
પણ -> 125744
માટે -> 125617
- -> 101993
એક -> 94749
કરી -> 93775
પર -> 92218
તે -> 82049
જ -> 81064
સાથે -> 80871
હતી -> 79548
તો -> 59450
હતો -> 56879
હતા -> 51945


In [9]:
for context, words in list(bigram.items())[:10]:

    print(
        context,
        "->",
        words.most_common(5)
    )

<s> -> [('આ', 75538), ('.', 30378), ('જો', 15244), ('તે', 14329), ('પરંતુ', 10548)]
૮૦ -> [('ટકા', 94), (',', 54), ('થી', 26), ('%', 19), (')', 19)]
થી -> [('વધુ', 2574), ('વધારે', 532), ('૬', 292), ('પણ', 214), ('12', 207)]
૧૦૦ -> [('ટકા', 169), ('થી', 143), ('જેટલા', 43), ('%', 34), ('કરોડ', 32)]
ના -> [('રોજ', 3229), ('પાડી', 451), ('હોય', 317), ('કારણે', 284), ('દિવસે', 273)]
ડઝન -> [('જેટલા', 34), ('જેટલી', 10), ('કરતા', 7), ('લોકો', 4), ('વર્ષ', 4)]
મળતા -> [('જ', 128), ('નથી', 109), ('હતા', 96), ('હોય', 81), ('આ', 37)]
હતા -> [('.', 36925), ('અને', 3775), (',', 3170), ('ત્યારે', 959), ('કે', 841)]
એ -> [('છે', 3762), ('.', 1762), ('જ', 1719), ('પણ', 1383), ('માટે', 637)]
હોલસેલ -> [('માર્કેટમાં', 6), ('વેપારી', 6), ('વેપારીઓ', 5), (',', 4), ('પ્રાઇઝ', 3)]


In [10]:
for context, words in list(trigram.items())[:10]:

    print(
        context,
        "->",
        words.most_common(5)
    )

('<s>', '<s>') -> [('આ', 75538), ('.', 30378), ('જો', 15244), ('તે', 14329), ('પરંતુ', 10548)]
('<s>', '૮૦') -> [(')', 16), ('ટકા', 7), (',', 5), ('ના', 4), ('હજાર', 4)]
('૮૦', 'થી') -> [('વધુ', 10), ('૧૦૦', 4), ('૮૫', 4), ('૯૦', 4), ('૧૫૦', 1)]
('થી', '૧૦૦') -> [('ટકા', 8), ('ના', 2), ('મી', 2), ('વય', 2), (',', 1)]
('૧૦૦', 'ના') -> [('કિલો', 4), ('સ્ટેમ્પ', 2), ('ડઝન', 1), ('કિલોનો', 1), ('ખાનામાં', 1)]
('ના', 'ડઝન') -> [('મળતા', 1)]
('ડઝન', 'મળતા') -> [('હતા', 1)]
('મળતા', 'હતા') -> [('.', 56), (',', 13), ('પરંતુ', 4), ('અને', 4), ('એ', 2)]
('હતા', 'એ') -> [('જ', 13), ('સમયે', 10), ('વખતે', 7), ('દરમ્યાન', 4), ('દરમિયાન', 4)]
('એ', 'હોલસેલ') -> [('માર્કેટમાં', 1)]


In [11]:
for context, words in list(quadrigram.items())[:10]:

    print(
        context,
        "->",
        words.most_common(5)
    )

('<s>', '<s>', '<s>') -> [('આ', 75538), ('.', 30378), ('જો', 15244), ('તે', 14329), ('પરંતુ', 10548)]
('<s>', '<s>', '૮૦') -> [(')', 16), ('ટકા', 7), (',', 5), ('ના', 4), ('હજાર', 4)]
('<s>', '૮૦', 'થી') -> [('૧૦૦', 1), ('વધુ', 1)]
('૮૦', 'થી', '૧૦૦') -> [('ના', 2), ('મીટર', 1), ('ફુટ', 1)]
('થી', '૧૦૦', 'ના') -> [('ડઝન', 1), ('કિલો', 1)]
('૧૦૦', 'ના', 'ડઝન') -> [('મળતા', 1)]
('ના', 'ડઝન', 'મળતા') -> [('હતા', 1)]
('ડઝન', 'મળતા', 'હતા') -> [('એ', 1)]
('મળતા', 'હતા', 'એ') -> [('હોલસેલ', 1), ('વિસ્તાર', 1)]
('હતા', 'એ', 'હોલસેલ') -> [('માર્કેટમાં', 1)]


In [12]:
def unigram_add1_probability(word, unigram, V):
    count = unigram.get(word, 0)
    total_words = sum(unigram.values())

    return (count + 1) / (total_words + V)


def bigram_add1_probability(previous_word, current_word, bigram, V):
    counts = bigram.get(previous_word)

    if counts is None:
        count = 0
        context_count = 0
    else:
        count = counts.get(current_word, 0)
        context_count = sum(counts.values())

    return (count + 1) / (context_count + V)


def trigram_add1_probability(
    previous_word1,
    previous_word2,
    current_word,
    trigram,
    V
):
    context = (previous_word1, previous_word2)
    counts = trigram.get(context)

    if counts is None:
        count = 0
        context_count = 0
    else:
        count = counts.get(current_word, 0)
        context_count = sum(counts.values())

    return (count + 1) / (context_count + V)


def quadrigram_add1_probability(
    previous_word1,
    previous_word2,
    previous_word3,
    current_word,
    quadrigram,
    V
):
    context = (
        previous_word1,
        previous_word2,
        previous_word3
    )

    counts = quadrigram.get(context)

    if counts is None:
        count = 0
        context_count = 0
    else:
        count = counts.get(current_word, 0)
        context_count = sum(counts.values())

    return (count + 1) / (context_count + V)


### Add-1 formula

For every n-gram:

$$P(w_i\mid context)=\frac{C(context,w_i)+1}{C(context)+V}$$

For an unseen n-gram, the numerator becomes 1, so its probability is non-zero.

For Bigram, the context count is $C(w_{i-1})$. For Trigram it is $C(w_{i-2},w_{i-1})$. For Quadrigram it is $C(w_{i-3},w_{i-2},w_{i-1})$.

In [13]:
sample_word = list(unigram.keys())[0]

print("Sample word:", sample_word)
print("Unigram Add-1 probability:", unigram_add1_probability(sample_word, unigram, V))
print("Bigram Add-1 probability:", bigram_add1_probability("<s>", sample_word, bigram, V))


Sample word: ૮૦
Unigram Add-1 probability: 3.150347578127087e-05
Bigram Add-1 probability: 4.589184284896561e-05


In [14]:
def most_common(counter):
    if not counter:
        return None
    return counter.most_common(1)[0][0]


def predict_bigram(previous_word, bigram):
    words = bigram.get(previous_word)
    return most_common(words)


def predict_trigram(previous_word1, previous_word2, trigram):
    words = trigram.get((previous_word1, previous_word2))
    return most_common(words)


def predict_quadrigram(
    previous_word1,
    previous_word2,
    previous_word3,
    quadrigram
):
    words = quadrigram.get(
        (previous_word1, previous_word2, previous_word3)
    )
    return most_common(words)


In [15]:
def predict_trigram(
    previous_word1,
    previous_word2,
    trigram
):

    words = trigram.get(
        (
            previous_word1,
            previous_word2
        )
    )

    return most_common(words)

In [16]:
def accuracy(sentences, order, bigram, trigram, quadrigram):
    correct = 0
    total = 0

    for tokens in sentences:
        tokens = list(tokens)

        # Bigram
        if order == 2:
            for i in range(len(tokens) - 1):
                previous_word = tokens[i]
                actual_word = tokens[i + 1]

                predicted_word = predict_bigram(
                    previous_word,
                    bigram
                )

                if predicted_word == actual_word:
                    correct += 1

                total += 1

        # Trigram
        elif order == 3:
            for i in range(len(tokens) - 2):
                previous_word1 = tokens[i]
                previous_word2 = tokens[i + 1]
                actual_word = tokens[i + 2]

                predicted_word = predict_trigram(
                    previous_word1,
                    previous_word2,
                    trigram
                )

                if predicted_word == actual_word:
                    correct += 1

                total += 1

        # Quadrigram
        elif order == 4:
            for i in range(len(tokens) - 3):
                previous_word1 = tokens[i]
                previous_word2 = tokens[i + 1]
                previous_word3 = tokens[i + 2]
                actual_word = tokens[i + 3]

                predicted_word = predict_quadrigram(
                    previous_word1,
                    previous_word2,
                    previous_word3,
                    quadrigram
                )

                if predicted_word == actual_word:
                    correct += 1

                total += 1

    if total == 0:
        return 0

    return correct / total


In [ ]:
bigram_dev_accuracy = accuracy(
    dev_df["tokens"], 2, bigram, trigram, quadrigram
)

trigram_dev_accuracy = accuracy(
    dev_df["tokens"], 3, bigram, trigram, quadrigram
)

quadrigram_dev_accuracy = accuracy(
    dev_df["tokens"], 4, bigram, trigram, quadrigram
)

print("Bigram Dev Accuracy:", bigram_dev_accuracy)
print("Trigram Dev Accuracy:", trigram_dev_accuracy)
print("Quadrigram Dev Accuracy:", quadrigram_dev_accuracy )


Bigram Dev Accuracy: 0.178442160127407
Trigram Dev Accuracy: 0.1772723732377911
Quadrigram Dev Accuracy: 0.10971207924116512


bigram_test_accuracy = accuracy(
    test_df["tokens"], 2, bigram, trigram, quadrigram
)

trigram_test_accuracy = accuracy(
    test_df["tokens"], 3, bigram, trigram, quadrigram
)

quadrigram_test_accuracy = accuracy(
    test_df["tokens"], 4, bigram, trigram, quadrigram
)

print("Bigram Test Accuracy:", bigram_test_accuracy)
print("Trigram Test Accuracy:", trigram_test_accuracy)
print("Quadrigram Test Accuracy:", quadrigram_test_accuracy)


In [18]:
word1 = "મને"
word2 = "ગુજરાતી"

prediction = predict_trigram(
    word1,
    word2,
    trigram
)

print(
    "Previous words:",
    word1,
    word2
)

print(
    "Predicted next word:",
    prediction
)

Previous words: મને ગુજરાતી
Predicted next word: વાનગીઓ


In [19]:
word1 = "મને"
word2 = "ગુજરાતી"
word3 = "ભાષા"

prediction = predict_quadrigram(
    word1,
    word2,
    word3,
    quadrigram
)

print(
    "Previous words:",
    word1,
    word2,
    word3
)

print(
    "Predicted next word:",
    prediction
)

Previous words: મને ગુજરાતી ભાષા
Predicted next word: None


## 7. Perplexity with Add-1 (Laplace) Smoothing

$$
PP =
\exp\left(
-\frac{1}{N}
\sum_{i=1}^{N}
\log P(w_i\mid context)
\right)
$$

Because Add-1 smoothing gives every possible prediction a non-zero probability, unseen n-grams do not make perplexity infinite.

**Lower perplexity is better.**


In [ ]:
bigram_dev_accuracy = accuracy(
    dev_df["tokens"],
    2,
    bigram,
    trigram,
    quadrigram
)

trigram_dev_accuracy = accuracy(
    dev_df["tokens"],
    3,
    bigram,
    trigram,
    quadrigram
)

quadrigram_dev_accuracy = accuracy(
    dev_df["tokens"],
    4,
    bigram,
    trigram,
    quadrigram
)


print(
    "Bigram Development Accuracy:",
    bigram_dev_accuracy 
)

print(
    "Trigram Development Accuracy:",
    trigram_dev_accuracy
)

print(
    "Quadrigram Development Accuracy:",
    quadrigram_dev_accuracy 
)

Bigram Development Accuracy: 0.178442160127407
Trigram Development Accuracy: 0.1772723732377911
Quadrigram Development Accuracy: 0.10971207924116512


In [ ]:
bigram_test_accuracy = accuracy(
    test_df["tokens"],
    2,
    bigram,
    trigram,
    quadrigram
)

trigram_test_accuracy = accuracy(
    test_df["tokens"],
    3,
    bigram,
    trigram,
    quadrigram
)

quadrigram_test_accuracy = accuracy(
    test_df["tokens"],
    4,
    bigram,
    trigram,
    quadrigram
)


print(
    "Bigram Test Accuracy:",
    bigram_test_accuracy 
)

print(
    "Trigram Test Accuracy:",
    trigram_test_accuracy
)

print(
    "Quadrigram Test Accuracy:",
    quadrigram_test_accuracy 
)

Bigram Test Accuracy: 0.17667638483965015
Trigram Test Accuracy: 0.1729517836142689
Quadrigram Test Accuracy: 0.10777683854606931


In [23]:
def get_ngrams(
    tokens,
    n
):

    tokens = (
        ["<s>"] * (n - 1)
        + list(tokens)
        + ["</s>"]
    )

    ngrams = []


    for i in range(
        len(tokens) - n + 1
    ):

        current_ngram = []


        for j in range(n):

            current_ngram.append(
                tokens[i + j]
            )


        ngrams.append(
            current_ngram
        )


    return ngrams

In [24]:
sentence = [
    "મને",
    "ગુજરાતી",
    "આવે"
]

print(
    get_ngrams(sentence, 2)
)

[['<s>', 'મને'], ['મને', 'ગુજરાતી'], ['ગુજરાતી', 'આવે'], ['આવે', '</s>']]


In [ ]:
def perplexity_add1(
    sentences,
    order,
    unigram,
    bigram,
    trigram,
    quadrigram,
    V
):
    log_probability = 0.0
    total = 0

    total_unigrams = sum(unigram.values())

    for tokens in sentences:
        ngrams = get_ngrams(tokens, order)

        for ngram in ngrams:
            word = ngram[-1]

            # Unigram
            if order == 1:
                count = unigram.get(word, 0)

                probability = (
                    count + 1
                ) / (
                    total_unigrams + V
                )

            # Bigram
            elif order == 2:
                context = ngram[0]
                counts = bigram.get(context)

                if counts is None:
                    count = 0
                    context_count = 0
                else:
                    count = counts.get(word, 0)
                    context_count = sum(counts.values())

                probability = (
                    count + 1
                ) / (
                    context_count + V
                )
            # Trigram
            elif order == 3:
                context = (ngram[0], ngram[1])
                counts = trigram.get(context)

                if counts is None:
                    count = 0
                    context_count = 0
                else:
                    count = counts.get(word, 0)
                    context_count = sum(counts.values())

                probability = (
                    count + 1
                ) / (
                    context_count + V
                )

            # Quadrigram
            elif order == 4:
                context = (
                    ngram[0],
                    ngram[1],
                    ngram[2]
                )
                counts = quadrigram.get(context)

                if counts is None:
                    count = 0
                    context_count = 0
                else:
                    count = counts.get(word, 0)
                    context_count = sum(counts.values())

                probability = (
                    count + 1
                ) / (
                    context_count + V
                )

            log_probability += math.log(probability)
            total += 1

    if total == 0:
        return 0.0

    return math.exp(
        -log_probability / total
    )


In [ ]:
# Development perplexity

unigram_dev_pp = perplexity_add1(
    dev_df["tokens"], 1, unigram, bigram, trigram, quadrigram, V
)

bigram_dev_pp = perplexity_add1(
    dev_df["tokens"], 2, unigram, bigram, trigram, quadrigram, V
)

trigram_dev_pp = perplexity_add1(
    dev_df["tokens"], 3, unigram, bigram, trigram, quadrigram, V
)

quadrigram_dev_pp = perplexity_add1(
    dev_df["tokens"], 4, unigram, bigram, trigram, quadrigram, V
)

print("Unigram Development Perplexity:", unigram_dev_pp)


print("Bigram Development Perplexity:", bigram_dev_pp)
print("Trigram Development Perplexity:", trigram_dev_pp)
print("Quadrigram Development Perplexity:", quadrigram_dev_pp)


Unigram Development Perplexity: 3334.714119841393
Bigram Development Perplexity: 11309.065012360605
Trigram Development Perplexity: 78358.30565219767
Quadrigram Development Perplexity: 185604.73708434074


In [ ]:
# Test perplexity

unigram_test_pp = perplexity_add1(
    test_df["tokens"], 1, unigram, bigram, trigram, quadrigram, V
)

bigram_test_pp = perplexity_add1(
    test_df["tokens"], 2, unigram, bigram, trigram, quadrigram, V
)

trigram_test_pp = perplexity_add1(
    test_df["tokens"], 3, unigram, bigram, trigram, quadrigram, V
)

quadrigram_test_pp = perplexity_add1(
    test_df["tokens"], 4, unigram, bigram, trigram, quadrigram, V
)

print("Unigram Test Perplexity:", unigram_test_pp)
print("Bigram Test Perplexity:", bigram_test_pp)
print("Trigram Test Perplexity:", trigram_test_pp)
print("Quadrigram Test Perplexity:", quadrigram_test_pp)


Unigram Test Perplexity: 3699.387186902719
Bigram Test Perplexity: 12687.66517816127
Trigram Test Perplexity: 84593.43797630808
Quadrigram Test Perplexity: 192596.3423526134


In [30]:
def show_bigram_prediction(
    word,
    bigram
):

    prediction = predict_bigram(
        word,
        bigram
    )

    print(
        "Input word:",
        word
    )

    print(
        "Predicted next word:",
        prediction
    )

In [31]:
show_bigram_prediction(
    "ગુજરાતી",
    bigram
)

Input word: ગુજરાતી
Predicted next word: ફિલ્મ


In [32]:
def show_trigram_prediction(
    word1,
    word2,
    trigram
):

    prediction = predict_trigram(
        word1,
        word2,
        trigram
    )

    print(
        "Input:",
        word1,
        word2
    )

    print(
        "Predicted next word:",
        prediction
    )

In [33]:
show_trigram_prediction(
    "મને",
    "ગુજરાતી",
    trigram
)

Input: મને ગુજરાતી
Predicted next word: વાનગીઓ


In [34]:
def show_quadrigram_prediction(
    word1,
    word2,
    word3,
    quadrigram
):

    prediction = predict_quadrigram(
        word1,
        word2,
        word3,
        quadrigram
    )

    print(
        "Input:",
        word1,
        word2,
        word3
    )

    print(
        "Predicted next word:",
        prediction
    )

In [35]:
show_quadrigram_prediction(
    "મને",
    "ગુજરાતી",
    "ભાષા",
    quadrigram
)

Input: મને ગુજરાતી ભાષા
Predicted next word: None
